# Busan Traffic AI PBL - 19_busan_model_interpretation.ipynb

부산시 교통량 예측 실습을 부산시 교통량 예측 프로젝트로 변경한 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
# =========================
# [셀 1] 라이브러리 로드
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

In [ ]:
# =========================
# [셀 2] 데이터 로드/병합/split
# =========================
X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

y_candidates = [c for c in df_y.columns if c != "date"]
y_col = y_candidates[0] if len(y_candidates) == 1 else ( "y" if "y" in y_candidates else y_candidates[-1] )

df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)

train_mask = (df["date"] >= "2022-01-01") & (df["date"] <= "2022-12-31")
test_mask  = (df["date"] >= "2023-02-01") & (df["date"] <= "2023-12-31")

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

feature_cols = [c for c in df.columns if c not in ["date", y_col]]
X_train, y_train = df_train[feature_cols], df_train[y_col]
X_test, y_test   = df_test[feature_cols], df_test[y_col]

In [ ]:
# =========================
# [셀 3] 전처리 + RF 학습
# =========================
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]), cat_cols),
    ],
    remainder="drop"
)

rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
pipe = Pipeline([("preprocess", preprocess), ("model", rf)])
pipe.fit(X_train, y_train)



In [ ]:
# =========================
# [셀 4] 해석 1: 내장 feature importance
# =========================
X_train_num = X_train[num_cols].copy()
X_test_num  = X_test[num_cols].copy()

pre_num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
Xtr = pre_num.fit_transform(X_train_num)
Xte = pre_num.transform(X_test_num)

rf_num = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf_num.fit(Xtr, y_train)

importances = rf_num.feature_importances_
idx = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(8,5))
plt.barh([num_cols[i] for i in idx][::-1], importances[idx][::-1])
plt.title("Top-11 Feature Importances (RF)")
plt.xlabel("importance")
plt.show()

In [ ]:
# =========================
# [셀 5] 해석 2: Permutation Importance
# - 어떤 feature를 섞었을 때 성능이 얼마나 나빠지는지로 중요도를 측정
# - 모델이 예측에 실제로 의존하는 정도를 더 직접적으로 보여줌
# =========================
perm = permutation_importance(
    rf_num, Xte, y_test,
    n_repeats=15,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

perm_mean = perm.importances_mean
idx = np.argsort(perm_mean)[::-1][:11]

plt.figure(figsize=(8,5))
plt.barh([num_cols[i] for i in idx][::-1], perm_mean[idx][::-1])
plt.title("Top-11 Permutation Importances (RMSE drop)")
plt.xlabel("importance (higher is more important)")
plt.show()

In [ ]:
# =========================
# [셀 2] 데이터 로드/병합/split
# =========================
X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

y_candidates = [c for c in df_y.columns if c != "date"]
y_col = y_candidates[0] if len(y_candidates) == 1 else ( "y" if "y" in y_candidates else y_candidates[-1] )

df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)

df = df.rename(columns={
    "평균기온(℃)": "avg_temp",
    "최저기온(℃)": "min_temp",
    "최고기온(℃)": "max_temp",
    "강수량(mm)": "precipitation"
})

train_mask = (df["date"] >= "2022-01-01") & (df["date"] <= "2022-12-31")
val_mask = (df["date"] >= "2023-01-01") & (df["date"] <= "2023-01-31")
test_mask  = (df["date"] >= "2023-02-01") & (df["date"] <= "2023-12-31")

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

feature_cols = [c for c in df.columns if c not in ["date", y_col]]
X_train, y_train = df_train[feature_cols], df_train[y_col]
X_val, y_val     = df_val[feature_cols], df_val[y_col]
X_test, y_test   = df_test[feature_cols], df_test[y_col]

In [ ]:
# =========================
# [셀 3] 전처리 + RF 학습
# =========================
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]), cat_cols),
    ],
    remainder="drop"
)

rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
pipe = Pipeline([("preprocess", preprocess), ("model", rf)])
pipe.fit(X_train, y_train)

In [ ]:
# =========================
# [셀 4] 해석 1: 내장 feature importance
# =========================
X_train_num = X_train[num_cols].copy()
X_test_num  = X_test[num_cols].copy()

pre_num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
Xtr = pre_num.fit_transform(X_train_num)
Xte = pre_num.transform(X_test_num)

rf_num = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf_num.fit(Xtr, y_train)

importances = rf_num.feature_importances_
idx = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(8,5))
plt.barh([num_cols[i] for i in idx][::-1], importances[idx][::-1])
plt.title("Top-15 Feature Importances (RF)")
plt.xlabel("importance")
plt.show()

In [ ]:
# =========================
# [셀 5] 해석 2: Permutation Importance
# - 어떤 feature를 섞었을 때 성능이 얼마나 나빠지는지로 중요도를 측정
# - 모델이 예측에 실제로 의존하는 정도를 더 직접적으로 보여줌
# =========================
perm = permutation_importance(
    rf_num, Xte, y_test,
    n_repeats=15,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

perm_mean = perm.importances_mean
idx = np.argsort(perm_mean)[::-1][:15]

plt.figure(figsize=(8,5))
plt.barh([num_cols[i] for i in idx][::-1], perm_mean[idx][::-1])
plt.title("Top-15 Permutation Importances (RMSE drop)")
plt.xlabel("importance (higher is more important)")
plt.show()

In [ ]:
# =========================
# [셀 1] 라이브러리
# =========================
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

In [ ]:
# =========================
# [셀 2] 평가 함수
# =========================
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # RMSE
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


In [ ]:
# =========================
# [셀 3] 전처리 + Ridge 모델 파이프라인
# =========================
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0))  # random_state 필요 없음
])


In [ ]:
# =========================
# [셀 4] 학습 및 성능 평가 (VAL/TEST)
# =========================
pipe.fit(X_train, y_train)

val_pred  = pipe.predict(X_val)
test_pred = pipe.predict(X_test)

r_val  = evaluate_regression(y_val, val_pred)
r_test = evaluate_regression(y_test, test_pred)

results = [{
    "val_MAE": r_val["MAE"],
    "val_RMSE": r_val["RMSE"],
    "val_R2": r_val["R2"],
    "test_MAE": r_test["MAE"],
    "test_RMSE": r_test["RMSE"],
    "test_R2": r_test["R2"],
}]

pd.DataFrame(results)


In [ ]:
# =========================
# [셀 5] Ridge 피처 중요도 1: 계수(coef) 기반
# =========================
ridge = pipe.named_steps["model"]

# X_train이 DataFrame이면 컬럼명이 그대로 중요도 이름이 됩니다.
feature_names = X_train.columns if hasattr(X_train, "columns") else [f"f{i}" for i in range(len(ridge.coef_))]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": ridge.coef_,
    "abs_coef": np.abs(ridge.coef_)
}).sort_values("abs_coef", ascending=False)

coef_df.head(30)


In [ ]:
# =========================
# [셀 6] (옵션) 계수 시각화 Top 11
# =========================
import matplotlib.pyplot as plt

topk = 11
tmp = coef_df.head(topk).iloc[::-1]  # 보기 좋게 역순

plt.figure(figsize=(8, 6))
plt.barh(tmp["feature"], tmp["coef"])
plt.xlabel("Ridge coefficient (after standardization)")
plt.ylabel("Feature")
plt.title(f"Top {topk} Ridge Coefficients")
plt.show()


In [ ]:
# =========================
# [셀 8] 피처 중요도 2: Permutation Importance (VAL 기준)
# =========================
perm = permutation_importance(
    pipe,
    X_val, y_val,
    n_repeats=20,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df.head(30)

In [ ]:
# =========================
# [셀 9] (옵션) Permutation Importance 시각화 Top 11
# =========================
import matplotlib.pyplot as plt

topk = 11
tmp = perm_df.head(topk).iloc[::-1]

plt.figure(figsize=(8, 6))
plt.barh(tmp["feature"], tmp["importance_mean"])
plt.xlabel("Permutation importance (decrease in score)")
plt.ylabel("Feature")
plt.title(f"Top {topk} Permutation Importances (Validation)")
plt.show()
